# Mutlistep trainer dev

In [1]:
# Standard library
import os
import sys
import logging
import datetime
from glob import glob
from typing import Any, Callable, Dict, TypedDict, List, Optional, Tuple, Union

# Third‑party
import yaml
import numpy as np
import xarray as xr

import torch
import torch.utils.data
from torch.utils.data import DistributedSampler, get_worker_info
from torchvision import transforms as tforms

# Local application
from credit.parser import credit_main_parser, training_data_check

In [2]:
from credit.datasets.wrf_multistep import WRF_MultiStep
from credit.transforms import Normalize_WRF, ToTensor_WRF

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
from credit.data import (
    generate_datetime,
    hour_to_nanoseconds,
    nanoseconds_to_year,
    extract_month_day_hour,
    find_common_indices,
    concat_and_reshape,
    reshape_only,
    get_forward_data,
    drop_var_from_dataset,
    find_key_for_number,
    next_six_hour,
    previous_six_hour_steps,
    encode_datetime64,
    filter_ds
)

In [5]:
from functools import partial

In [6]:
# Logging setup
logger = logging.getLogger(__name__)

# single node steup
rank = 0
world_size = 1

In [7]:
config_name = '/glade/work/ksha/DWC_runs/CONUS_GP_multi/model_multi.yml'
# Read YAML file
with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [8]:
conf = credit_main_parser(conf, parse_training=True, parse_predict=False, print_summary=True)

Upper-air variables: ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q']
Surface variables: ['WRF_SP', 'WRF_T2', 'WRF_TD2', 'WRF_U10', 'WRF_V10']
Dynamic forcing variables: []
Diagnostic variables: ['WRF_precip', 'WRF_PWAT', 'WRF_radar_composite', 'WRF_TCC', 'WRF_MLCAPE', 'WRF_OLR']
Forcing variables: []
Static variables: ['z_norm', 'var_norm', 'lu_norm', 'LANDMASK']


## dataset dev

In [9]:
Array = Union[np.ndarray, xr.DataArray]

class Sample_WRF(TypedDict):
    # Shape: batch_size, seq_length, lat, lon, lev
    WRF_input: Array
    WRF_target: Array
    boundary_input: Array
    time_encode: Array
    datetime_index: Array

In [10]:
# pick a year
train_years_range = [2018, 2020]
valid_years_range = [2018, 2020]

# ============================================================================== #
# WRF domain interior
# ---------------------------
# param_interior
# param_outside
# ----------------------------
#     varname_upper_air
#     varname_surface
#     varname_dyn_forcing
#     varname_forcing
#     varname_static
#     varname_diagnostic
#     filenames
#     filename_surface
#     filename_dyn_forcing
#     filename_forcing
#     filename_static
#     filename_diagnostic
#     history_len
#     forecast_len
# ============================================================================== #

param_interior = {}
param_outside = {}
# --------------- #
# upper air files
upper_files = sorted(glob(conf["data"]["save_loc"]))
upper_files_outside = sorted(glob(conf["data"]['boundary']["save_loc"]))

# --------------- #
# surface files
if ('surface_variables' in conf['data']) and (len(conf['data']['surface_variables']) > 0):
    list_surf_ds = sorted(glob(conf["data"]["save_loc_surface"]))
else:
    list_surf_ds = None

list_surf_ds_outside = sorted(glob(conf["data"]['boundary']["save_loc_surface"]))

# --------------- #
# dyn forcing files
if ('dynamic_forcing_variables' in conf['data']) and (len(conf['data']['dynamic_forcing_variables']) > 0):
    list_dyn_forcing_ds = sorted(glob(conf["data"]["save_loc_dynamic_forcing"]))
else:
    list_dyn_forcing_ds = None

# --------------- #
# diagnostic files
if ('diagnostic_variables' in conf['data']) and (len(conf['data']['diagnostic_variables']) > 0):
    list_diag_ds = sorted(glob(conf["data"]["save_loc_diagnostic"]))
else:
    list_diag_ds = None

# convert year info to str for file name search
train_years = [str(year) for year in range(train_years_range[0], train_years_range[1])]
valid_years = [str(year) for year in range(valid_years_range[0], valid_years_range[1])]

# Filter the files for training / validation
train_files = [file for file in upper_files if any(year in file for year in train_years)]
valid_files = [file for file in upper_files if any(year in file for year in valid_years)]

train_files_outside = [file for file in upper_files_outside if any(year in file for year in train_years)]
valid_files_outside = [file for file in upper_files_outside if any(year in file for year in valid_years)]

if list_surf_ds is not None:
    train_list_surf_ds = [file for file in list_surf_ds if any(year in file for year in train_years)]
    valid_list_surf_ds = [file for file in list_surf_ds if any(year in file for year in valid_years)]
else:
    train_list_surf_ds = None
    valid_list_surf_ds = None

train_list_surf_ds_outside = [file for file in list_surf_ds_outside if any(year in file for year in train_years)]
valid_list_surf_ds_outside = [file for file in list_surf_ds_outside if any(year in file for year in valid_years)]

if list_dyn_forcing_ds is not None:
    train_list_dyn_forcing_ds = [file for file in list_dyn_forcing_ds if any(year in file for year in train_years)]
    valid_list_dyn_forcing_ds = [file for file in list_dyn_forcing_ds if any(year in file for year in valid_years)]

else:
    train_list_dyn_forcing_ds = None
    valid_list_dyn_forcing_ds = None

if list_diag_ds is not None:
    train_list_diag_ds = [file for file in list_diag_ds if any(year in file for year in train_years)]
    valid_list_diag_ds = [file for file in list_diag_ds if any(year in file for year in valid_years)]
else:
    train_list_diag_ds = None
    valid_list_diag_ds = None
    
param_interior['varname_upper_air'] = conf['data']['variables']
param_interior['varname_surface'] = conf['data']['surface_variables']
param_interior['varname_dyn_forcing'] = conf['data']['dynamic_forcing_variables']
param_interior['varname_forcing'] = conf['data']['forcing_variables']
param_interior['varname_static'] = conf['data']['static_variables']
param_interior['varname_diagnostic'] = conf['data']['diagnostic_variables']
param_interior['filename_forcing'] = conf['data']['save_loc_forcing']
param_interior['filename_static'] = conf['data']['save_loc_static']

param_outside['varname_upper_air'] = conf["data"]['boundary']['variables']
param_outside['varname_surface'] = conf["data"]['boundary']['surface_variables']
# --------------------------------------------------- #
is_train = False

# separate training set and validation set cases
if is_train:
    param_interior['filenames'] = train_files
    param_interior['filename_surface'] = train_list_surf_ds
    param_interior['filename_dyn_forcing'] = train_list_dyn_forcing_ds
    param_interior['filename_diagnostic'] = train_list_diag_ds
    param_interior['history_len'] = conf["data"]["history_len"]
    param_interior['forecast_len'] = conf["data"]["forecast_len"]

    param_outside['filenames'] = train_files_outside
    param_outside['filename_surface'] = train_list_surf_ds_outside
    param_outside['history_len'] = conf["data"]['boundary']["history_len"]
    param_outside['forecast_len'] = conf["data"]['boundary']["forecast_len"]
    
    name = "training"
    
else:
    param_interior['filenames'] = valid_files
    param_interior['filename_surface'] = valid_list_surf_ds
    param_interior['filename_dyn_forcing'] = valid_list_dyn_forcing_ds
    param_interior['filename_diagnostic'] = valid_list_diag_ds
    param_interior['history_len'] = conf["data"]["valid_history_len"]
    param_interior['forecast_len'] = conf["data"]["valid_forecast_len"]
    
    param_outside['filenames'] = valid_files_outside
    param_outside['filename_surface'] = valid_list_surf_ds_outside
    param_outside['history_len'] = conf["data"]['boundary']["history_len"]
    param_outside['forecast_len'] = conf["data"]['boundary']["forecast_len"]
    
    name = 'validation'
    

In [24]:
conf['data']['boundary']['variables']
conf['data']['boundary']['surface_variables']

['MSL', 'VAR_2T', 'VAR_10U', 'VAR_10V']

In [11]:
to_tensor_scaler = ToTensor_WRF(conf)
normalizer = Normalize_WRF(conf)
transforms = tforms.Compose([normalizer, to_tensor_scaler])

In [19]:
dataset = WRF_MultiStep(
    param_interior,
    param_outside,
    transform=None,
)

In [20]:
batch_single = dataset.__getitem__(3)

In [18]:
batch_single.keys()

dict_keys(['x_forcing_static', 'x_surf', 'x', 'y_diag', 'y_surf', 'y', 'x_boundary', 'x_surf_boundary', 'x_time_encode', 'index', 'datetime', 'forecast_step', 'stop_forecast'])

In [21]:
batch_single['boundary_input']

<xarray.Dataset>
Dimensions:    (time: 2, level: 11, latitude: 88, longitude: 112)
Coordinates:
  * latitude   (latitude) float64 23.25 23.5 23.75 24.0 ... 44.5 44.75 45.0
  * level      (level) float64 1e+03 950.0 850.0 700.0 ... 200.0 100.0 50.0
  * longitude  (longitude) float64 250.2 250.5 250.8 251.0 ... 277.5 277.8 278.0
  * time       (time) datetime64[ns] 2018-01-01 2018-01-01T06:00:00
Data variables:
    Q          (time, level, latitude, longitude) float32 dask.array<chunksize=(1, 11, 88, 112), meta=np.ndarray>
    T          (time, level, latitude, longitude) float32 dask.array<chunksize=(1, 11, 88, 112), meta=np.ndarray>
    U          (time, level, latitude, longitude) float32 dask.array<chunksize=(1, 11, 88, 112), meta=np.ndarray>
    V          (time, level, latitude, longitude) float32 dask.array<chunksize=(1, 11, 88, 112), meta=np.ndarray>
    MSL        (time, latitude, longitude) float32 dask.array<chunksize=(1, 88, 112), meta=np.ndarray>
    VAR_10U    (time, latitude, longitude) float32 dask.array<chunksize=(1, 88, 112), meta=np.ndarray>
    VAR_10V    (time, latitude, longitude) float32 dask.array<chunksize=(1, 88, 112), meta=np.ndarray>
    VAR_2T     (time, latitude, longitude) float32 dask.array<chunksize=(1, 88, 112), meta=np.ndarray>
Attributes:
    CONVERSION_DATE:      Sat Sep  7 08:13:03 MDT 2019
    CONVERSION_PLATFORM:  Linux casper12 3.10.0-693.21.1.el7.x86_64 #1 SMP We...
    Conventions:          CF-1.6
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NCO:                  netCDF Operators version 4.7.4 (http://nco.sf.net)
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    history:              Sat Sep  7 08:13:39 2019: ncks -4 --ppc default=7 e...

In [51]:
batch_single['WRF_input']

<xarray.Dataset>
Dimensions:      (time: 1, bottom_top: 16, south_north: 336, west_east: 336)
Coordinates:
  * bottom_top   (bottom_top) float32 0.0 1.0 2.0 3.0 ... 12.0 13.0 14.0 15.0
  * south_north  (south_north) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
  * time         (time) datetime64[ns] 2018-01-01
  * west_east    (west_east) float32 0.0 1.0 2.0 3.0 ... 332.0 333.0 334.0 335.0
Data variables: (12/14)
    WRF_P        (time, bottom_top, south_north, west_east) float32 8.817e+04...
    WRF_U        (time, bottom_top, south_north, west_east) float32 -7.956 .....
    WRF_V        (time, bottom_top, south_north, west_east) float32 1.095 ......
    WRF_T        (time, bottom_top, south_north, west_east) float32 284.5 ......
    WRF_Q        (time, bottom_top, south_north, west_east) float32 0.007742 ...
    WRF_SP       (time, south_north, west_east) float32 8.844e+04 ... 1.015e+05
    ...           ...
    WRF_U10      (time, south_north, west_east) float32 -6.984 -9.534 ... 1.915
    WRF_V10      (time, south_north, west_east) float32 0.9671 -1.221 ... -2.513
    LANDMASK     (time, south_north, west_east) float32 1.0 1.0 1.0 ... 1.0 1.0
    lu_norm      (time, south_north, west_east) float32 0.4118 0.4118 ... 0.7059
    var_norm     (time, south_north, west_east) float32 0.7052 ... -0.3643
    z_norm       (time, south_north, west_east) float32 1.239 1.313 ... -0.407

In [52]:
batch_single['WRF_target']

<xarray.Dataset>
Dimensions:              (time: 1, bottom_top: 16, south_north: 336,
                          west_east: 336)
Coordinates:
  * bottom_top           (bottom_top) float32 0.0 1.0 2.0 3.0 ... 13.0 14.0 15.0
  * south_north          (south_north) float32 0.0 1.0 2.0 ... 333.0 334.0 335.0
  * time                 (time) datetime64[ns] 2018-01-01T01:00:00
  * west_east            (west_east) float32 0.0 1.0 2.0 ... 333.0 334.0 335.0
Data variables: (12/16)
    WRF_P                (time, bottom_top, south_north, west_east) float32 8...
    WRF_U                (time, bottom_top, south_north, west_east) float32 -...
    WRF_V                (time, bottom_top, south_north, west_east) float32 0...
    WRF_T                (time, bottom_top, south_north, west_east) float32 2...
    WRF_Q                (time, bottom_top, south_north, west_east) float32 0...
    WRF_SP               (time, south_north, west_east) float32 8.858e+04 ......
    ...                   ...
    WRF_precip           (time, south_north, west_east) float32 0.0 ... 0.002062
    WRF_PWAT             (time, south_north, west_east) float32 0.0237 ... 0....
    WRF_radar_composite  (time, south_north, west_east) float32 0.0 0.0 ... 0.0
    WRF_TCC              (time, south_north, west_east) float32 0.0 0.0 ... 1.0
    WRF_MLCAPE           (time, south_north, west_east) float32 0.0 0.0 ... 0.0
    WRF_OLR              (time, south_north, west_east) float32 253.4 ... 188.5

In [10]:
to_tensor_scaler = ToTensor_WRF(conf)
normalizer = Normalize_WRF(conf)
transforms = tforms.Compose([normalizer, to_tensor_scaler])

In [11]:
dataset = WRF_Dataset(
    param_interior,
    param_outside,
    transform=transforms,
)

In [12]:
batch_single = dataset.__getitem__(333)

In [13]:
print(batch_single['x'].shape)
print(batch_single['x_boundary'].shape)
print(batch_single['x_surf_boundary'].shape)

torch.Size([1, 5, 16, 336, 336])
torch.Size([2, 4, 11, 88, 112])
torch.Size([2, 4, 88, 112])


In [14]:
print((batch_single['x'].min(), batch_single['x'].max()))
print((batch_single['y'].min(), batch_single['y'].max()))

print((batch_single['x_boundary'].min(), batch_single['x_boundary'].max()))
print((batch_single['x_surf_boundary'].min(), batch_single['x_surf_boundary'].max()))

(tensor(-4.3625), tensor(4.6441))
(tensor(-4.3637), tensor(4.7427))
(tensor(-4.5811), tensor(3.8562))
(tensor(-3.3377), tensor(3.6726))


In [19]:
print(batch_single['x'].shape)
print(batch_single['x_boundary'].shape)
print(batch_single['x_surf_boundary'].shape)

torch.Size([1, 5, 16, 336, 336])
torch.Size([2, 4, 11, 88, 112])
torch.Size([2, 4, 88, 112])


In [20]:
print((batch_single['x'].min(), batch_single['x'].max()))
print((batch_single['y'].min(), batch_single['y'].max()))

print((batch_single['x_boundary'].min(), batch_single['x_boundary'].max()))
print((batch_single['x_surf_boundary'].min(), batch_single['x_surf_boundary'].max()))

(tensor(-4.3625), tensor(4.6441))
(tensor(-4.3637), tensor(4.7427))
(tensor(-4.5811), tensor(3.8562))
(tensor(-3.3377), tensor(3.6726))


(tensor(-35.6056), tensor(11.6433))


In [12]:
# for i in range(9999):
#     batch_single = dataset.__getitem__(i)
#     flag1 = batch_single['x'].shape == torch.Size([1, 4, 12, 336, 336])
#     flag2 = batch_single['x_boundary'].shape == torch.Size([1, 4, 11, 88, 112])
#     flag3 = batch_single['x_surf_boundary'].shape == torch.Size([1, 4, 88, 112])
#     flag4 = batch_single['x_time_encode'].shape == torch.Size([12])
    
#     if flag1 & flag2 & flag3 & flag4:
#         pass;
#     else:
#         print(i)
#         raise


# has_nan = any(
#     torch.isnan(tensor).any() 
#     for tensor in batch_single.values() 
#     if isinstance(tensor, torch.Tensor)
# )

# print("Contains NaNs:", has_nan)

In [13]:
batch_single = dataset.__getitem__(333)

In [14]:
batch_single.keys()

dict_keys(['WRF_input', 'WRF_target', 'boundary_input', 'time_encode', 'datetime_index', 'index'])

In [47]:
def worker(
    tuple_index: Tuple[int, int],
    WRF_file_indices: Dict[str, List[int]],
    list_upper_ds: List[Any],
    list_surf_ds: Optional[List[Any]],
    list_dyn_forcing_ds: Optional[List[Any]],
    list_diag_ds: Optional[List[Any]],
    xarray_forcing: Optional[Any],
    xarray_static: Optional[Any],
    history_len: int,
    forecast_len: int,
    list_upper_ds_outside: Optional[List[Any]],
    list_surf_ds_outside: Optional[List[Any]],
    outside_file_year_range: Optional[List[Any]],
    outside_file_indices: Optional[List[Any]],
    history_len_outside: int,
    transform: Optional[Callable],
) -> Dict[str, Any]:


    index, ind_start_current_step = tuple_index

    try:
        # ========================================================================== #
        # cross-year indices --> the index of the year + indices within that year

        # select the ind_file based on the iter index
        ind_file = find_key_for_number(ind_start_current_step, WRF_file_indices)

        # get the ind within the current file
        ind_start = WRF_file_indices[ind_file][1]
        ind_start_in_file = ind_start_current_step - ind_start

        # handle out-of-bounds
        ind_largest = len(list_upper_ds[int(ind_file)]["time"]) - (history_len + forecast_len + 1)
        
        if ind_start_in_file > ind_largest:
            ind_start_in_file = ind_largest
            
        # ========================================================================== #
        # subset xarray on time dimension

        ind_end_in_file = ind_start_in_file + history_len + forecast_len
        
        ## WRF_file_subset: a xarray dataset that contains training input and target (for the current batch)
        WRF_subset = list_upper_ds[int(ind_file)].isel(
            time=slice(ind_start_in_file, ind_end_in_file + 1)
        )
        
        # ========================================================================== #
        # merge surface into the dataset

        if list_surf_ds:
            ## subset surface variables
            surface_subset = list_surf_ds[int(ind_file)].isel(
                time=slice(ind_start_in_file, ind_end_in_file + 1)
            )

            ## merge upper-air and surface here:
            WRF_subset = WRF_subset.merge(
                surface_subset
            )  # <-- lazy merge, upper and surface both not loaded

        # ==================================================== #
        # split WRF_subset into training inputs and targets
        #   + merge with dynamic forcing, forcing, and static

        # the ind_end of the WRF_subset
        # ind_end_time = len(WRF_subset["time"])

        # datetiem information as int number (used in some normalization methods)
        datetime_as_number = WRF_subset.time.values.astype("datetime64[s]").astype(int)

        # ==================================================== #
        # xarray dataset as input
        ## WRF_input: the final input

        WRF_input = WRF_subset.isel(time=slice(0, history_len, 1)).load()

        # ========================================================================== #
        # merge dynamic forcing inputs
        if list_dyn_forcing_ds:
            dyn_forcing_subset = list_dyn_forcing_ds[int(ind_file)].isel(
                time=slice(ind_start_in_file, ind_end_in_file + 1)
            )
            dyn_forcing_subset = dyn_forcing_subset.isel(
                time=slice(0, history_len, 1)
            ).load()

            WRF_input = WRF_input.merge(dyn_forcing_subset)

        # ========================================================================== #
        # merge forcing inputs
        if xarray_forcing:
            # ------------------------------------------------------------------------------- #
            # matching month, day, hour between forcing and upper air [time]
            # this approach handles leap year forcing file and non-leap-year upper air file
            month_day_forcing = extract_month_day_hour(np.array(xarray_forcing["time"]))
            month_day_inputs = extract_month_day_hour(np.array(WRF_input["time"]))
            # indices to subset
            ind_forcing, _ = find_common_indices(month_day_forcing, month_day_inputs)
            forcing_subset_input = xarray_forcing.isel(time=ind_forcing)
            # forcing and upper air have different years but the same mon/day/hour
            # safely replace forcing time with upper air time
            forcing_subset_input["time"] = WRF_input["time"]
            # ------------------------------------------------------------------------------- #

            # merge
            WRF_input = WRF_input.merge(forcing_subset_input)

        # ========================================================================== #
        # merge static inputs
        if xarray_static:
            # expand static var on time dim
            N_time_dims = len(WRF_subset["time"])
            static_subset_input = xarray_static.expand_dims(dim={"time": N_time_dims})
            # assign coords 'time'
            static_subset_input = static_subset_input.assign_coords({"time": WRF_subset["time"]})
            # slice, update time and merge
            static_subset_input = static_subset_input.isel(time=slice(0, history_len, 1))
            static_subset_input["time"] = WRF_input["time"]
            WRF_input = WRF_input.merge(static_subset_input)
            
        # ==================================================== #
        # xarray dataset as target
        ## WRF_target: the final target
        
        WRF_target = WRF_subset.isel(time=slice(history_len, history_len+1, 1)).load()

        ## merge diagnoisc input here:
        if list_diag_ds:
            # subset diagnostic variables
            diagnostic_subset = list_diag_ds[int(ind_file)].isel(
                time=slice(ind_start_in_file, ind_end_in_file + 1)
            )

            diagnostic_subset = diagnostic_subset.isel(
                time=slice(history_len, history_len+1, 1)
            ).load()

            # merge into the target dataset
            WRF_target = WRF_target.merge(diagnostic_subset)


        # ==================================================== #
        # handle boundary files
        # ==================================================== #
        time_boundary = WRF_target['time'].values[0] # <--- assuming single time value here
        time_round = next_six_hour(time_boundary)
        
        if history_len_outside == 1:    
            time_year = int(np.datetime_as_string(time_round, unit="Y"))
            ind_year = time_year - outside_file_year_range[0]
            ind_date = np.searchsorted(outside_file_indices[str(ind_year)], time_round)
            ds_upper_outside = list_upper_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1))
            ds_surf_outside = list_surf_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1))
            ds_outside = xr.merge([ds_upper_outside, ds_surf_outside])

        else:
            list_ds_upper_outside_slice = []
            list_ds_surf_outside_slice = []
            
            for i_time_backward in range(history_len_outside):
                time_round_loop = previous_six_hour_steps(time_round, i_time_backward)
                time_year = int(np.datetime_as_string(time_round_loop, unit="Y"))
                ind_year = time_year - outside_file_year_range[0]
                ind_date = np.searchsorted(outside_file_indices[str(ind_year)], time_round_loop)
                list_ds_upper_outside_slice.append(list_upper_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1)))
                list_ds_surf_outside_slice.append(list_surf_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1)))
                
            ds_upper_outside = xr.concat(list_ds_upper_outside_slice[::-1], dim='time') # ::-1 so the latest time is the last
            ds_surf_outside = xr.concat(list_ds_surf_outside_slice[::-1], dim='time')
            ds_outside = xr.merge([ds_upper_outside, ds_surf_outside])

        # ==================================================== #
        # encode datetime input
        # ==================================================== #
        t0 = WRF_input['time'].values
        t1 = WRF_target['time'].values
        t2 = ds_outside['time'].values
        time_encode = encode_datetime64(np.concatenate([t0, t1, t2]))
        
        sample = Sample_WRF(
            WRF_input=WRF_input,
            WRF_target=WRF_target,
            boundary_input=ds_outside,
            time_encode=time_encode,
            datetime_index=datetime_as_number,
        )
        
        # ==================================== #
        # data normalization
        if transform:
            sample = transform(sample)
            
        sample["index"] = index
        sample["datetime"] = [
            int(WRF_input.time.values[0].astype("datetime64[s]").astype(int)),
            int(WRF_target.time.values[0].astype("datetime64[s]").astype(int)),
        ]

    except Exception as e:
        logger.error(f"Error processing index {tuple_index}: {e}")
        raise

    return sample

In [38]:
class WRF_MultiStep(torch.utils.data.Dataset):
    
    def __init__(
        self,
        param_interior,
        param_outside,
        transform=None,
        seed=42,
        rank=0,
        world_size=1,
        max_forecast_len=None,
    ):

        # ========================================================== #
        # WRF domain variable and filename info
        varname_upper_air = param_interior['varname_upper_air']
        varname_surface = param_interior['varname_surface']
        varname_dyn_forcing = param_interior['varname_dyn_forcing']
        varname_forcing = param_interior['varname_forcing']
        varname_static = param_interior['varname_static']
        varname_diagnostic = param_interior['varname_diagnostic']
        filenames = param_interior['filenames']
        filename_surface = param_interior['filename_surface']
        filename_dyn_forcing = param_interior['filename_dyn_forcing']
        filename_forcing = param_interior['filename_forcing']
        filename_static = param_interior['filename_static']
        filename_diagnostic = param_interior['filename_diagnostic']
        # ----------------------------------------------------------- #
        
        list_upper_ds = []
        list_surf_ds = []
        list_dyn_forcing_ds = []
        list_diag_ds = []
        filenames = sorted(filenames)
        
        all_ds = [get_forward_data(fn) for fn in filenames]
        
        # 1. Upper‐air
        list_upper_ds = [filter_ds(ds, varname_upper_air) for ds in all_ds]
        
        # 2. Surface
        if filename_surface:
            list_surf_ds = [filter_ds(ds, varname_surface) for ds in all_ds]
        else:
            list_surf_ds = False

        # 3. Dynamic forcing
        if filename_dyn_forcing:
            list_dyn_forcing_ds = [filter_ds(ds, varname_dyn_forcing) for ds in all_ds]
        else:
            list_dyn_forcing_ds = False
            
        # 4. Diagnostics
        if filename_diagnostic:
            list_diag_ds = [filter_ds(ds, varname_diagnostic) for ds in all_ds]
        else:
            list_diag_ds = False

        self.list_upper_ds = list_upper_ds
        self.list_surf_ds = list_surf_ds
        self.list_dyn_forcing_ds = list_dyn_forcing_ds
        self.list_diag_ds = list_diag_ds
        
        self.history_len = param_interior['history_len']
        self.forecast_len = param_interior['forecast_len']
        self.seed = seed
        self.rank = rank
        self.world_size = world_size
        self.total_seq_len = self.history_len + self.forecast_len
        
        # -------------------------------------------------------------------------- #
        # get sample indices from WRF upper-air files:
        ind_start = 0
        self.WRF_file_indices = {}  # <------ change
        for ind_file, WRF_file_xarray in enumerate(self.list_upper_ds):
            # [number of samples, ind_start, ind_end]
            self.WRF_file_indices[str(ind_file)] = [
                len(WRF_file_xarray["time"]),
                ind_start,
                ind_start + len(WRF_file_xarray["time"]),
            ]
            ind_start += len(WRF_file_xarray["time"]) + 1

        # -------------------------------------------------------------------------- #
        # forcing file
        self.filename_forcing = filename_forcing
        if self.filename_forcing is not None:
            # drop variables if they are not in the config
            ds = get_forward_data(filename_forcing)
            ds_forcing = drop_var_from_dataset(ds, varname_forcing).load()
            self.xarray_forcing = ds_forcing
        else:
            self.xarray_forcing = False

        # -------------------------------------------------------------------------- #
        # static file
        self.filename_static = filename_static
        if self.filename_static is not None:
            # drop variables if they are not in the config
            ds = get_forward_data(filename_static)
            ds_static = drop_var_from_dataset(ds, varname_static).load()
            self.xarray_static = ds_static
        else:
            self.xarray_static = False

        # ========================================================== #
        # boundary variable and filename info
        varname_upper_air_outside = param_outside['varname_upper_air']
        varname_surface_outside = param_outside['varname_surface']
        filenames_outside = param_outside['filenames']
        filename_surface_outside = param_outside['filename_surface']
        # ----------------------------------------------------------- #
        # collecting xr.datasets
        list_upper_ds_outside = []
        list_surf_ds_outside = []
        filenames_outside = sorted(filenames_outside)
        
        for fn_outside in filenames_outside:
            # drop variables if they are not in the config
            ds_outside = get_forward_data(filename=fn_outside)
            ds_upper_outside = drop_var_from_dataset(ds_outside, varname_upper_air_outside)

            if filename_surface_outside is not None:
                ds_surf_outside = drop_var_from_dataset(ds_outside, varname_surface_outside)
                list_surf_ds_outside.append(ds_surf_outside)
            else:
                self.list_surf_ds_outside = False

            list_upper_ds_outside.append(ds_upper_outside)

        self.list_upper_ds_outside = list_upper_ds_outside
        self.list_surf_ds_outside = list_surf_ds_outside
        self.history_len_outside = param_outside['history_len']
        self.forecast_len_outside = param_outside['forecast_len']
        self.total_seq_len = self.history_len_outside + self.forecast_len_outside
        # -------------------------------------------------------------------------- #
        # get sample indices from boundary upper-air files:
        self.outside_file_year_range = [
            int(np.datetime_as_string(self.list_upper_ds_outside[0]["time"][0].values, unit="Y")),
            int(np.datetime_as_string(self.list_upper_ds_outside[-1]["time"][0].values, unit="Y"))
        ]
        
        self.outside_file_indices = {}  # <------ change
        for ind_file, outside_file_xarray in enumerate(self.list_upper_ds_outside):
            self.outside_file_indices[str(ind_file)] = outside_file_xarray["time"].values
            
        # ========================================================== #
        
        self.transform = transform
        self.rng = np.random.default_rng(seed=seed)
        self.max_forecast_len = max_forecast_len
        
        self.worker = partial(
            worker,
            WRF_file_indices=self.WRF_file_indices,
            list_upper_ds=self.list_upper_ds,
            list_surf_ds=self.list_surf_ds,
            list_dyn_forcing_ds=self.list_dyn_forcing_ds,
            list_diag_ds=self.list_diag_ds,
            xarray_forcing=self.xarray_forcing,
            xarray_static=self.xarray_static,
            history_len=self.history_len,
            forecast_len=self.forecast_len,
            list_upper_ds_outside=self.list_upper_ds_outside,
            list_surf_ds_outside=self.list_surf_ds_outside,
            outside_file_year_range=self.outside_file_year_range,
            outside_file_indices=self.outside_file_indices,
            history_len_outside=self.history_len_outside,
            transform=self.transform,
        )

        self.total_length = len(self.WRF_file_indices)
        self.current_epoch = None
        self.forecast_step_count = 0
        self.current_index = None
        self.initial_index = None

    def __post_init__(self):
        # Total sequence length of each sample.
        self.total_seq_len = self.history_len + self.forecast_len

    def __len__(self):
        # compute the total number of length
        total_len = 0
        for ERA5_xarray in self.all_files:
            total_len += len(ERA5_xarray["time"]) - self.total_seq_len + 1
        return total_len

    def set_epoch(self, epoch):
        self.current_epoch = epoch
        self.forecast_step_count = 0
        self.current_index = None
        self.initial_index = None

    def __getitem__(self, index):
        if (self.forecast_step_count == self.forecast_len + 1) or (self.current_index is None):
            # We've completed the last forecast or we're starting for the first time
            # Start a new forecast using the sampler index
            self.current_index = index  # self._get_random_start_index()
            self.forecast_step_count = 0
            index = self.current_index
            self.initial_index = self.current_index
        else:
            # Ignore the sampler index and continue the forecast
            self.current_index += 1
            index = self.current_index

        # Worker process
        sample = self.worker((self.initial_index, index))

        # assign sample index
        sample["forecast_step"] = self.forecast_step_count + 1
        sample["index"] = index
        sample["stop_forecast"] = self.forecast_step_count == self.forecast_len

        # update the step count
        self.forecast_step_count += 1

        return sample

In [ ]:
sampler = RepeatingIndexSampler(
            dataset,
            forecast_len=forecast_len,
            num_replicas=world_size,
            rank=rank,
            seed=seed,
            shuffle=is_train,
        )
        dataloader = DataLoader(
            dataset,
            batch_size=1,
            shuffle=False,
            sampler=sampler,
            pin_memory=True,
            persistent_workers=False,
            num_workers=1,  # set to one so prefetch is working
            prefetch_factor=prefetch_factor,
        )

# Workspace

## dataset dev

In [19]:
_PERIOD_NS = int(np.timedelta64(6, "h") / np.timedelta64(1, "ns"))

def next_six_hour(dt):
    ns = dt.astype("int64")
    out = (ns // _PERIOD_NS + 1) * _PERIOD_NS
    return out.astype("datetime64[ns]")

def previous_six_hour_steps(time_pick, step):
    """
    Given a datetime64[ns] time_pick, compute time_pick - step * 6 hours.
    """
    return time_pick - np.timedelta64(6 * step, 'h')

In [20]:
# t0 = np.datetime64("2018-01-14T22:00:00")
# next_six_hour(t0)

In [21]:
def encode_datetime64(dt_array):
    dt_array = np.atleast_1d(dt_array).astype('datetime64[ns]')
    dt_s = dt_array.astype('datetime64[s]')

    # Time components
    seconds_in_day = 86400
    seconds_since_midnight = (dt_s - dt_s.astype('datetime64[D]')).astype('timedelta64[s]').astype(int)
    hour = seconds_since_midnight / 3600.0

    # Day of year
    year_start = dt_s.astype('datetime64[Y]')
    day_of_year = (dt_s - year_start).astype('timedelta64[D]').astype(int) + 1

    # Cyclical encodings
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    doy_sin = np.sin(2 * np.pi * day_of_year / 365.25)
    doy_cos = np.cos(2 * np.pi * day_of_year / 365.25)

    return np.concatenate((hour_sin, hour_cos, doy_sin, doy_cos), axis=0)

In [35]:
class WRF_Dataset(torch.utils.data.Dataset):
    '''
    WRF/regional model Pytorch Dataset class
    '''

    def __init__(
        self,
        param_interior,
        param_outside,
        transform=None,
        seed=42,
    ):
        # ========================================================== #
        # WRF domain variable and filename info
        varname_upper_air = param_interior['varname_upper_air']
        varname_surface = param_interior['varname_surface']
        varname_dyn_forcing = param_interior['varname_dyn_forcing']
        varname_forcing = param_interior['varname_forcing']
        varname_static = param_interior['varname_static']
        varname_diagnostic = param_interior['varname_diagnostic']
        filenames = param_interior['filenames']
        filename_surface = param_interior['filename_surface']
        filename_dyn_forcing = param_interior['filename_dyn_forcing']
        filename_forcing = param_interior['filename_forcing']
        filename_static = param_interior['filename_static']
        filename_diagnostic = param_interior['filename_diagnostic']
        # ----------------------------------------------------------- #
        # collecting xr.datasets
        list_upper_ds = []
        list_surf_ds = []
        list_dyn_forcing_ds = []
        list_diag_ds = []
        filenames = sorted(filenames)
        
        for fn in filenames:
            # drop variables if they are not in the config
            ds = get_forward_data(filename=fn)
            ds_upper = drop_var_from_dataset(ds, varname_upper_air)

            if filename_surface is not None:
                ds_surf = drop_var_from_dataset(ds, varname_surface)
                list_surf_ds.append(ds_surf)
            else:
                self.list_surf_ds = False

            if filename_dyn_forcing is not None:
                ds_dyn = drop_var_from_dataset(ds, varname_dyn_forcing)
                list_dyn_forcing_ds.append(ds_dyn)
            else:
                self.list_dyn_forcing_ds = False

            if filename_diagnostic is not None:
                ds_diag = drop_var_from_dataset(ds, varname_diagnostic)
                list_diag_ds.append(ds_diag)
            else:
                self.list_diag_ds = False

            list_upper_ds.append(ds_upper)

        self.list_upper_ds = list_upper_ds
        self.list_surf_ds = list_surf_ds
        self.list_dyn_forcing_ds = list_dyn_forcing_ds
        self.list_diag_ds = list_diag_ds
        self.history_len = param_interior['history_len']
        self.forecast_len = param_interior['forecast_len']
        self.total_seq_len = self.history_len + self.forecast_len
        # -------------------------------------------------------------------------- #
        # get sample indices from WRF upper-air files:
        ind_start = 0
        self.WRF_file_indices = {}  # <------ change
        for ind_file, WRF_file_xarray in enumerate(self.list_upper_ds):
            # [number of samples, ind_start, ind_end]
            self.WRF_file_indices[str(ind_file)] = [
                len(WRF_file_xarray["time"]),
                ind_start,
                ind_start + len(WRF_file_xarray["time"]),
            ]
            ind_start += len(WRF_file_xarray["time"]) + 1

        # -------------------------------------------------------------------------- #
        # forcing file
        self.filename_forcing = filename_forcing
        if self.filename_forcing is not None:
            # drop variables if they are not in the config
            ds = get_forward_data(filename_forcing)
            ds_forcing = drop_var_from_dataset(ds, varname_forcing).load()
            self.xarray_forcing = ds_forcing
        else:
            self.xarray_forcing = False

        # -------------------------------------------------------------------------- #
        # static file
        self.filename_static = filename_static
        if self.filename_static is not None:
            # drop variables if they are not in the config
            ds = get_forward_data(filename_static)
            ds_static = drop_var_from_dataset(ds, varname_static).load()
            self.xarray_static = ds_static
        else:
            self.xarray_static = False
        
        
        # ========================================================== #
        # boundary variable and filename info
        varname_upper_air_outside = param_outside['varname_upper_air']
        varname_surface_outside = param_outside['varname_surface']
        filenames_outside = param_outside['filenames']
        filename_surface_outside = param_outside['filename_surface']
        # ----------------------------------------------------------- #
        # collecting xr.datasets
        list_upper_ds_outside = []
        list_surf_ds_outside = []
        filenames_outside = sorted(filenames_outside)
        
        for fn_outside in filenames_outside:
            # drop variables if they are not in the config
            ds_outside = get_forward_data(filename=fn_outside)
            ds_upper_outside = drop_var_from_dataset(ds_outside, varname_upper_air_outside)

            if filename_surface_outside is not None:
                ds_surf_outside = drop_var_from_dataset(ds_outside, varname_surface_outside)
                list_surf_ds_outside.append(ds_surf_outside)
            else:
                self.list_surf_ds_outside = False

            list_upper_ds_outside.append(ds_upper_outside)

        self.list_upper_ds_outside = list_upper_ds_outside
        self.list_surf_ds_outside = list_surf_ds_outside
        self.history_len_outside = param_outside['history_len']
        self.forecast_len_outside = param_outside['forecast_len']
        self.total_seq_len = self.history_len_outside + self.forecast_len_outside
        # -------------------------------------------------------------------------- #
        # get sample indices from boundary upper-air files:
        self.outside_file_year_range = [
            int(np.datetime_as_string(self.list_upper_ds_outside[0]["time"][0].values, unit="Y")),
            int(np.datetime_as_string(self.list_upper_ds_outside[-1]["time"][0].values, unit="Y"))
        ]
        
        self.outside_file_indices = {}  # <------ change
        for ind_file, outside_file_xarray in enumerate(self.list_upper_ds_outside):
            self.outside_file_indices[str(ind_file)] = outside_file_xarray["time"].values
            
        # ========================================================== #
        # shared by the two domains
        self.transform = transform
        self.rng = np.random.default_rng(seed=seed)
        
    def __post_init__(self):
        # Total sequence length of each sample.
        self.total_seq_len = self.history_len + self.forecast_len

    def __len__(self):
        # compute the total number of length
        total_len = 0
        for WRF_file_xarray in self.list_upper_ds:
            total_len += len(WRF_file_xarray["time"]) - self.total_seq_len + 1
        return total_len

    def __getitem__(self, index):
        # ========================================================================== #
        # cross-year indices --> the index of the year + indices within that year

        # select the ind_file based on the iter index
        ind_file = find_key_for_number(index, self.WRF_file_indices)

        # get the ind within the current file
        ind_start = self.WRF_file_indices[ind_file][1]
        ind_start_in_file = index - ind_start

        # handle out-of-bounds
        ind_largest = len(self.list_upper_ds[int(ind_file)]["time"]) - (
            self.history_len + self.forecast_len + 1
        )
        if ind_start_in_file > ind_largest:
            ind_start_in_file = ind_largest
            
        # ========================================================================== #
        # subset xarray on time dimension

        ind_end_in_file = ind_start_in_file + self.history_len + self.forecast_len

        ## WRF_file_subset: a xarray dataset that contains training input and target (for the current batch)
        WRF_subset = self.list_upper_ds[int(ind_file)].isel(
            time=slice(ind_start_in_file, ind_end_in_file + 1)
        )  # .load() NOT load into memory

        # ========================================================================== #
        # merge surface into the dataset

        if self.list_surf_ds:
            ## subset surface variables
            surface_subset = self.list_surf_ds[int(ind_file)].isel(
                time=slice(ind_start_in_file, ind_end_in_file + 1)
            )  # .load() NOT load into memory

            ## merge upper-air and surface here:
            WRF_subset = WRF_subset.merge(
                surface_subset
            )  # <-- lazy merge, ERA5 and surface both not loaded

        # ==================================================== #
        # split WRF_subset into training inputs and targets
        #   + merge with dynamic forcing, forcing, and static

        # the ind_end of the WRF_subset
        ind_end_time = len(WRF_subset["time"])

        # datetiem information as int number (used in some normalization methods)
        datetime_as_number = WRF_subset.time.values.astype("datetime64[s]").astype(int)

        # ==================================================== #
        # xarray dataset as input
        ## WRF_input: the final input

        WRF_input = WRF_subset.isel(
            time=slice(0, self.history_len, 1)
        ).load()  # <-- load into memory

        # ========================================================================== #
        # merge dynamic forcing inputs
        if self.list_dyn_forcing_ds:
            dyn_forcing_subset = self.list_dyn_forcing_ds[int(ind_file)].isel(
                time=slice(ind_start_in_file, ind_end_in_file + 1)
            )
            dyn_forcing_subset = dyn_forcing_subset.isel(
                time=slice(0, self.history_len, 1)
            ).load()  # <-- load into memory

            WRF_input = WRF_input.merge(dyn_forcing_subset)

        # ========================================================================== #
        # merge forcing inputs
        if self.xarray_forcing:
            # ------------------------------------------------------------------------------- #
            # matching month, day, hour between forcing and upper air [time]
            # this approach handles leap year forcing file and non-leap-year upper air file
            month_day_forcing = extract_month_day_hour(np.array(self.xarray_forcing["time"]))
            month_day_inputs = extract_month_day_hour(np.array(WRF_input["time"]))
            # indices to subset
            ind_forcing, _ = find_common_indices(month_day_forcing, month_day_inputs)
            forcing_subset_input = self.xarray_forcing.isel(time=ind_forcing)
            # forcing and upper air have different years but the same mon/day/hour
            # safely replace forcing time with upper air time
            forcing_subset_input["time"] = WRF_input["time"]
            # ------------------------------------------------------------------------------- #

            # merge
            WRF_input = WRF_input.merge(forcing_subset_input)

        # ========================================================================== #
        # merge static inputs
        if self.xarray_static:
            # expand static var on time dim
            N_time_dims = len(WRF_subset["time"])
            static_subset_input = self.xarray_static.expand_dims(dim={"time": N_time_dims})
            # assign coords 'time'
            static_subset_input = static_subset_input.assign_coords({"time": WRF_subset["time"]})
            # slice, update time and merge
            static_subset_input = static_subset_input.isel(time=slice(0, self.history_len, 1))
            static_subset_input["time"] = WRF_input["time"]
            WRF_input = WRF_input.merge(static_subset_input)

        # ==================================================== #
        # xarray dataset as target
        ## WRF_target: the final target
        
        WRF_target = WRF_subset.isel(time=slice(self.history_len, ind_end_time, 1)).load() # <-- load into memory

        ## merge diagnoisc input here:
        if self.list_diag_ds:
            # subset diagnostic variables
            diagnostic_subset = self.list_diag_ds[int(ind_file)].isel(
                time=slice(ind_start_in_file, ind_end_in_file + 1)
            )

            diagnostic_subset = diagnostic_subset.isel(
                time=slice(self.history_len, ind_end_time, 1)
            ).load()  # <-- load into memory

            # merge into the target dataset
            WRF_target = WRF_target.merge(diagnostic_subset)

        # ==================================================== #
        # handle boundary files
        # ==================================================== #
        time_boundary = WRF_target['time'].values[0] # <--- assuming single time value here
        time_round = next_six_hour(time_boundary)
        
        if self.history_len_outside == 1:    
            time_year = int(np.datetime_as_string(time_round, unit="Y"))
            ind_year = time_year - self.outside_file_year_range[0]
            ind_date = np.searchsorted(self.outside_file_indices[str(ind_year)], time_round)
            ds_upper_outside = self.list_upper_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1))
            ds_surf_outside = self.list_surf_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1))
            ds_outside = xr.merge([ds_upper_outside, ds_surf_outside])

        else:
            list_ds_upper_outside_slice = []
            list_ds_surf_outside_slice = []
            
            for i_time_backward in range(self.history_len_outside):
                time_round_loop = previous_six_hour_steps(time_round, i_time_backward)
                time_year = int(np.datetime_as_string(time_round_loop, unit="Y"))
                ind_year = time_year - self.outside_file_year_range[0]
                ind_date = np.searchsorted(self.outside_file_indices[str(ind_year)], time_round_loop)
                list_ds_upper_outside_slice.append(self.list_upper_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1)))
                list_ds_surf_outside_slice.append(self.list_surf_ds_outside[ind_year].isel(time=slice(ind_date, ind_date+1)))
                
            ds_upper_outside = xr.concat(list_ds_upper_outside_slice[::-1], dim='time') # ::-1 so the latest time is the last
            ds_surf_outside = xr.concat(list_ds_surf_outside_slice[::-1], dim='time')
            ds_outside = xr.merge([ds_upper_outside, ds_surf_outside])
            
        # ==================================================== #
        # encode datetime input
        # ==================================================== #
        t0 = WRF_input['time'].values
        t1 = WRF_target['time'].values
        t2 = ds_outside['time'].values
        time_encode = encode_datetime64(np.concatenate([t0, t1, t2]))
        
        # pipe xarray datasets to the sampler
        sample = Sample(
            WRF_input=WRF_input,
            WRF_target=WRF_target,
            boundary_input=ds_outside,
            time_encode=time_encode,
            datetime_index=datetime_as_number,
        )

        # ==================================== #
        # data normalization
        if self.transform:
            sample = self.transform(sample)

        # assign sample index
        sample["index"] = index

        return sample

## transform dev

In [55]:
class Normalize_WRF:
    def __init__(self, conf):
        self.mean_ds = xr.open_dataset(conf["data"]["mean_path"]).load()
        self.std_ds = xr.open_dataset(conf["data"]["std_path"]).load()
        
        varnames_all = conf["data"]["all_varnames"]
        
        self.mean_tensors = {}
        self.std_tensors = {}

        for var in varnames_all:
            mean_array = self.mean_ds[var].values
            std_array = self.std_ds[var].values
            # convert to tensor
            self.mean_tensors[var] = torch.tensor(mean_array)
            self.std_tensors[var] = torch.tensor(std_array)

        # Get levels and upper air variables
        self.levels = conf["data"]["levels"]  # It was conf['model']['levels']
        self.varname_upper_air = conf["data"]["variables"]
        self.num_upper_air = len(self.varname_upper_air) * self.levels

        # Identify the existence of other variables
        self.flag_surface = ("surface_variables" in conf["data"]) and (
            len(conf["data"]["surface_variables"]) > 0
        )
        self.flag_dyn_forcing = ("dynamic_forcing_variables" in conf["data"]) and (
            len(conf["data"]["dynamic_forcing_variables"]) > 0
        )
        self.flag_diagnostic = ("diagnostic_variables" in conf["data"]) and (
            len(conf["data"]["diagnostic_variables"]) > 0
        )
        self.flag_forcing = ("forcing_variables" in conf["data"]) and (
            len(conf["data"]["forcing_variables"]) > 0
        )
        self.flag_static = ("static_variables" in conf["data"]) and (
            len(conf["data"]["static_variables"]) > 0
        )

        # Get surface varnames
        if self.flag_surface:
            self.varname_surface = conf["data"]["surface_variables"]
            self.num_surface = len(self.varname_surface)

        # Get dynamic forcing varnames
        if self.flag_dyn_forcing:
            self.varname_dyn_forcing = conf["data"]["dynamic_forcing_variables"]
            self.num_dyn_forcing = len(self.varname_dyn_forcing)

        # Get diagnostic varnames
        if self.flag_diagnostic:
            self.varname_diagnostic = conf["data"]["diagnostic_variables"]
            self.num_diagnostic = len(self.varname_diagnostic)

        # Get forcing varnames
        if self.flag_forcing:
            self.varname_forcing = conf["data"]["forcing_variables"]
        else:
            self.varname_forcing = []

        # Get static varnames:
        if self.flag_static:
            self.varname_static = conf["data"]["static_variables"]
        else:
            self.varname_static = []

        if self.flag_forcing or self.flag_static:
            self.has_forcing_static = True
            self.num_static = len(self.varname_static)
            self.num_forcing = len(self.varname_forcing)
            self.num_forcing_static = self.num_static + self.num_forcing
            self.varname_forcing_static = self.varname_forcing + self.varname_static
            self.static_first = conf["data"]["static_first"]
        else:
            self.has_forcing_static = False
            
        logger.info("WRF domain z-score parameters loaded")
        
        # ======================================================================= #
        # boundary condition data handling
        # ======================================================================= #
        self.mean_ds_outside = xr.open_dataset(conf["data"]['boundary']["mean_path"]).load()
        self.std_ds_outside = xr.open_dataset(conf["data"]['boundary']["std_path"]).load()
        
        varnames_all_outside = conf["data"]['boundary']["all_varnames"]
        
        self.mean_tensors_outside = {}
        self.std_tensors_outside = {}
        
        for var in varnames_all_outside:
            mean_array = self.mean_ds_outside[var].values
            std_array = self.std_ds_outside[var].values
            # convert to tensor
            self.mean_tensors_outside[var] = torch.tensor(mean_array)
            self.std_tensors_outside[var] = torch.tensor(std_array)

        # Get levels and upper air variables
        self.levels_outside = conf["data"]['boundary']["levels"]
        self.varname_upper_air_outside = conf["data"]['boundary']["variables"]
        self.num_upper_air_outside = len(self.varname_upper_air_outside) * self.levels_outside
        
        self.flag_surface_outside = ("surface_variables" in conf["data"]['boundary']) and (
            len(conf["data"]['boundary']["surface_variables"]) > 0
        )
        
        # Get surface varnames
        if self.flag_surface:
            self.varname_surface_outside = conf["data"]['boundary']["surface_variables"]
            self.num_surface_outside = len(self.varname_surface_outside)

        logger.info("Boundary domain z-score parameters loaded")


    def __call__(self, sample: Sample, inverse: bool = False) -> Sample:
        if inverse:
            # Inverse transformation
            return self.inverse_transform(sample)
        else:
            # Transformation
            return self.transform(sample)

    def transform_array(self, x: torch.Tensor) -> torch.Tensor:
        """
        This function applies to y_pred, so there won't be boundary input, forcing, and static variables.
        """
        # Get the current device
        device = x.device

        # Subset upper air
        tensor_upper_air = x[:, : self.num_upper_air, :, :]
        transformed_upper_air = tensor_upper_air.clone()

        # Surface variables
        if self.flag_surface:
            tensor_surface = x[:, self.num_upper_air : (self.num_upper_air + self.num_surface), :, :]
            transformed_surface = tensor_surface.clone()

        # y_pred does not have dynamic_forcing, skip this var type

        # Diagnostic variables (the very last of the stack)
        if self.flag_diagnostic:
            tensor_diagnostic = x[:, -self.num_diagnostic :, :, :]
            transformed_diagnostic = tensor_diagnostic.clone()

        # Standardize upper air variables
        # Upper air variable structure: var 1 [all levels] --> var 2 [all levels]
        k = 0
        for name in self.varname_upper_air:
            mean_tensor = self.mean_tensors[name].to(device)
            std_tensor = self.std_tensors[name].to(device)
            
            for level in range(self.levels):
                var_mean = mean_tensor[level]
                var_std = std_tensor[level]
                transformed_upper_air[:, k] = (tensor_upper_air[:, k] - var_mean) / var_std
                k += 1

        # Standardize surface variables
        if self.flag_surface:
            for k, name in enumerate(self.varname_surface):
                var_mean = self.mean_tensors[name].to(device)
                var_std = self.std_tensors[name].to(device)
                transformed_surface[:, k] = (tensor_surface[:, k] - var_mean) / var_std

        # Standardize diagnostic variables
        if self.flag_diagnostic:
            for k, name in enumerate(self.varname_diagnostic):
                var_mean = self.mean_tensors[name].to(device)
                var_std = self.std_tensors[name].to(device)
                transformed_diagnostic[:, k] = (transformed_diagnostic[:, k] - var_mean) / var_std

        # Concatenate everything
        if self.flag_surface:
            if self.flag_diagnostic:
                
                transformed_x = torch.cat((
                    transformed_upper_air, 
                    transformed_surface, 
                    transformed_diagnostic,), dim=1)
                
            else:
                transformed_x = torch.cat((transformed_upper_air, transformed_surface), dim=1)
        else:
            if self.flag_diagnostic:
                transformed_x = torch.cat((transformed_upper_air, transformed_diagnostic), dim=1)
            else:
                transformed_x = transformed_upper_air

        return transformed_x.to(device)

    def transform(self, sample: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        """
        This function transforms training batches
            - forcing & static don't need to be transformed; users should transform them and save them to the file
            - other variables need to be transformed
        """
        normalized_sample = {}
        if self.has_forcing_static:
            for key, value in sample.items():
                # key: 'historical_ERA5_images', 'target_ERA5_images'
                # value: the xarray datasets
                if isinstance(value, xr.Dataset):
                    # training input
                    if key == "WRF_input":
                        # get all the input vars
                        varname_inputs = value.keys()

                        # loop through dataset variables, handle forcing and static differently
                        for varname in varname_inputs:
                            
                            # if forcing and static skip it, otherwise do z-score
                            if (varname in self.varname_forcing_static) is False:
                                value[varname] = (value[varname] - self.mean_ds[varname]) / self.std_ds[varname]
                                
                        # put transformed xr.Dataset to the output dictionary
                        normalized_sample[key] = value

                    # WRF target fields
                    elif key == "WRF_target":
                        normalized_sample[key] = (value - self.mean_ds) / self.std_ds

                    # boundary inputs
                    elif key == "boundary_input":
                        normalized_sample[key] = (value - self.mean_ds_outside) / self.std_ds_outside
                elif key == 'time_encode':
                    normalized_sample[key] = value

        # if there's no forcing / static
        else:
            for key, value in sample.items():
                if isinstance(value, xr.Dataset):
                    # WRF domain
                    if key == "WRF_input" or key == "WRF_target":
                        normalized_sample[key] = (value - self.mean_ds) / self.std_ds
                        
                    # boundary inputs
                    elif key == "boundary_input":
                        normalized_sample[key] = (value - self.mean_ds_outside) / self.std_ds_outside
                        
                elif key == 'time_encode':
                    normalized_sample[key] = value
                        
        return normalized_sample

    def inverse_transform(self, x: torch.Tensor) -> torch.Tensor:
        """
        This function applies to y_pred, so there won't be dynamic forcing, forcing, and static vars
        """
        # Get the current device
        device = x.device

        # Subset upper air
        tensor_upper_air = x[:, : self.num_upper_air, :, :]
        transformed_upper_air = tensor_upper_air.clone()

        # Surface variables
        if self.flag_surface:
            tensor_surface = x[:, self.num_upper_air : (self.num_upper_air + self.num_surface), :, :]
            transformed_surface = tensor_surface.clone()

        # Diagnostic variables (the very last of the stack)
        if self.flag_diagnostic:
            tensor_diagnostic = x[:, -self.num_diagnostic :, :, :]
            transformed_diagnostic = tensor_diagnostic.clone()

        # Reverse upper air variables
        k = 0
        for name in self.varname_upper_air:
            mean_tensor = self.mean_tensors[name].to(device)
            std_tensor = self.std_tensors[name].to(device)
            for level in range(self.levels):
                mean = mean_tensor[level]
                std = std_tensor[level]
                transformed_upper_air[:, k] = tensor_upper_air[:, k] * std + mean
                k += 1

        # Reverse surface variables
        if self.flag_surface:
            for k, name in enumerate(self.varname_surface):
                mean = self.mean_tensors[name].to(device)
                std = self.std_tensors[name].to(device)
                transformed_surface[:, k] = tensor_surface[:, k] * std + mean

        # Reverse diagnostic variables
        if self.flag_diagnostic:
            for k, name in enumerate(self.varname_diagnostic):
                mean = self.mean_tensors[name].to(device)
                std = self.std_tensors[name].to(device)
                transformed_diagnostic[:, k] = transformed_diagnostic[:, k] * std + mean

        # Concatenate everything
        if self.flag_surface:
            if self.flag_diagnostic:
                transformed_x = torch.cat((
                    transformed_upper_air,
                    transformed_surface,
                    transformed_diagnostic,), dim=1)
            else:
                transformed_x = torch.cat((transformed_upper_air, transformed_surface), dim=1)
        else:
            if self.flag_diagnostic:
                transformed_x = torch.cat((transformed_upper_air, transformed_diagnostic), dim=1)
            else:
                transformed_x = transformed_upper_air

        return transformed_x.to(device)

In [61]:
class ToTensor_WRF:
    def __init__(self, conf):
        self.conf = conf

        # =============================================== #
        self.output_dtype = torch.float32
        # ============================================== #

        self.hist_len = int(conf["data"]["history_len"])
        self.for_len = int(conf["data"]["forecast_len"])

        # identify the existence of other variables
        self.flag_surface = ("surface_variables" in conf["data"]) and (
            len(conf["data"]["surface_variables"]) > 0
        )
        self.flag_dyn_forcing = ("dynamic_forcing_variables" in conf["data"]) and (
            len(conf["data"]["dynamic_forcing_variables"]) > 0
        )
        self.flag_diagnostic = ("diagnostic_variables" in conf["data"]) and (
            len(conf["data"]["diagnostic_variables"]) > 0
        )
        self.flag_forcing = ("forcing_variables" in conf["data"]) and (
            len(conf["data"]["forcing_variables"]) > 0
        )
        self.flag_static = ("static_variables" in conf["data"]) and (
            len(conf["data"]["static_variables"]) > 0
        )

        self.varname_upper_air = conf["data"]["variables"]

        # get surface varnames
        if self.flag_surface:
            self.varname_surface = conf["data"]["surface_variables"]

        # get dynamic forcing varnames
        self.num_forcing_static = 0

        if self.flag_dyn_forcing:
            self.varname_dyn_forcing = conf["data"]["dynamic_forcing_variables"]
            self.num_forcing_static += len(self.varname_dyn_forcing)
        else:
            self.varname_dyn_forcing = []

        # get diagnostic varnames
        if self.flag_diagnostic:
            self.varname_diagnostic = conf["data"]["diagnostic_variables"]

        # get forcing varnames
        if self.flag_forcing:
            self.varname_forcing = conf["data"]["forcing_variables"]
            self.num_forcing_static += len(self.varname_forcing)
        else:
            self.varname_forcing = []

        # get static varnames:
        if self.flag_static:
            self.varname_static = conf["data"]["static_variables"]
            self.num_forcing_static += len(self.varname_static)
        else:
            self.varname_static = []

        if self.flag_forcing or self.flag_static:
            self.has_forcing_static = True
            # ======================================================================================== #
            # forcing variable first (new models) vs. static variable first (some old models)
            # this flag makes sure that the class is compatible with some old CREDIT models
            self.flag_static_first = ("static_first" in conf["data"]) and (conf["data"]["static_first"])
            # ======================================================================================== #
        else:
            self.has_forcing_static = False

        # ======================================================================= #
        # boundary condition data handling
        # ======================================================================= #
        self.hist_len_outside = int(conf["data"]['boundary']["history_len"])
        self.for_len_outside = int(conf["data"]['boundary']["forecast_len"])
        
        self.flag_surface_outside = ("surface_variables" in conf["data"]['boundary']) and (
            len(conf["data"]['boundary']["surface_variables"]) > 0
        )

        self.varname_upper_air_outside = conf["data"]['boundary']["variables"]

        # get surface varnames
        if self.flag_surface_outside:
            self.varname_surface_outside = conf["data"]['boundary']["surface_variables"]


    def __call__(self, sample: Sample) -> Sample:
        return_dict = {}

        for key, value in sample.items():
            ## if DataArray
            if isinstance(value, xr.DataArray):
                var_value = value.values

            ## if Dataset
            elif isinstance(value, xr.Dataset):

                # WRF domain ds to numpy conversion
                if key == 'WRF_input' or key == 'WRF_target':
                    
                    # organize upper-air vars
                    list_vars_upper_air = []
                    for var_name in self.varname_upper_air:
                        var_value = value[var_name].values
                        list_vars_upper_air.append(var_value)
    
                    # [num_vars, hist_len, num_levels, lat, lon]
                    numpy_vars_upper_air = np.array(list_vars_upper_air)  
    
                    # organize surface vars
                    if self.flag_surface:
                        
                        list_vars_surface = []
                        for var_name in self.varname_surface:
                            var_value = value[var_name].values
                            list_vars_surface.append(var_value)
    
                        # [num_surf_vars, hist_len, lat, lon]
                        numpy_vars_surface = np.array(list_vars_surface)  
                        
                    # organize forcing and static (input only)
                    if self.has_forcing_static or self.flag_dyn_forcing:
                        
                        # enter this scope if one of the (dyn_forcing, folrcing, static) exists
                        if self.flag_static_first:
                            varname_forcing_static = (self.varname_static + self.varname_dyn_forcing + self.varname_forcing)
                        else:
                            varname_forcing_static = (self.varname_dyn_forcing + self.varname_forcing + self.varname_static)
    
                        if key == "WRF_input":
                            list_vars_forcing_static = []
                            for var_name in varname_forcing_static:
                                var_value = value[var_name].values
                                list_vars_forcing_static.append(var_value)
                            numpy_vars_forcing_static = np.array(list_vars_forcing_static)
    
                    # organize diagnostic vars (target only)
                    if self.flag_diagnostic:
                        if key == "WRF_target":
                            list_vars_diagnostic = []
                            for var_name in self.varname_diagnostic:
                                var_value = value[var_name].values
                                list_vars_diagnostic.append(var_value)
                            numpy_vars_diagnostic = np.array(list_vars_diagnostic)

                # ================================================================= #
                # boundary domain ds to numpy conversion
                # ================================================================= #
                elif key == 'boundary_input':
                    list_vars_upper_air_outside = []
                    for var_name in self.varname_upper_air_outside:
                        var_value = value[var_name].values
                        list_vars_upper_air_outside.append(var_value)
    
                    # [num_vars, hist_len, num_levels, lat, lon]
                    numpy_vars_upper_air_outside = np.array(list_vars_upper_air_outside)

                    # organize surface vars
                    if self.flag_surface_outside:
                        
                        list_vars_surface_outside = []
                        for var_name in self.varname_surface_outside:
                            var_value = value[var_name].values
                            list_vars_surface_outside.append(var_value)
                            
                        # [num_surf_vars, hist_len, lat, lon]
                        numpy_vars_surface_outside = np.array(list_vars_surface_outside)  
                        
            ## if numpy
            else:
                var_value = value

            # WRF domain tensor conversion
            if key == "WRF_input" or key == "WRF_target":
                # ---------------------------------------------------------------------- #
                # ToTensor: upper-air varialbes
                ## produces [time, upper_var, level, lat, lon]
                ## np.hstack concatenates the second dim (axis=1)
                x_upper_air = np.hstack([np.expand_dims(var_upper_air, axis=1) for var_upper_air in numpy_vars_upper_air])
                x_upper_air = torch.as_tensor(x_upper_air)
    
                # ---------------------------------------------------------------------- #
                # ToTensor: surface variables
                if self.flag_surface:
                    # this line produces [surface_var, time, lat, lon]
                    x_surf = torch.as_tensor(numpy_vars_surface).squeeze()
    
                    if len(x_surf.shape) == 4:
                        # permute: [surface_var, time, lat, lon] --> [time, surface_var, lat, lon]
                        x_surf = x_surf.permute(1, 0, 2, 3)
                        
                    # separate single variable vs. single history_len
                    elif len(x_surf.shape) == 3:
                        if len(self.varname_surface) > 1:
                            # single time, multi-vars
                            x_surf = x_surf.unsqueeze(0)
                        else:
                            # multi-time, single vars
                            x_surf = x_surf.unsqueeze(1)
                            
                    else:
                        # num_var=1, time=1, only has lat, lon
                        x_surf = x_surf.unsqueeze(0).unsqueeze(0)
    
                if key == "WRF_input":
                    # ToTensor: forcing and static
                    if self.has_forcing_static:
                        # this line produces [forcing_var, time, lat, lon]
                        x_static = torch.as_tensor(numpy_vars_forcing_static).squeeze()
    
                        if len(x_static.shape) == 4:
                            # permute: [forcing_var, time, lat, lon] --> [time, forcing_var, lat, lon]
                            x_static = x_static.permute(1, 0, 2, 3)
    
                        elif len(x_static.shape) == 3:
                            if self.num_forcing_static > 1:
                                # single time, multi-vars
                                x_static = x_static.unsqueeze(0)
                            else:
                                # multi-time, single vars
                                x_static = x_static.unsqueeze(1)
                        else:
                            # num_var=1, time=1, only has lat, lon
                            x_static = x_static.unsqueeze(0).unsqueeze(0)
                            # x_static = x_static.unsqueeze(1)
    
                        return_dict["x_forcing_static"] = x_static.type(self.output_dtype)
    
                    if self.flag_surface:
                        return_dict["x_surf"] = x_surf.type(self.output_dtype)
    
                    return_dict["x"] = x_upper_air.type(self.output_dtype)
    
                elif key == "WRF_target":
                    # ---------------------------------------------------------------------- #
                    # ToTensor: diagnostic
                    if self.flag_diagnostic:
                        # this line produces [forcing_var, time, lat, lon]
                        y_diag = torch.as_tensor(numpy_vars_diagnostic).squeeze()
    
                        if len(y_diag.shape) == 4:
                            # permute: [diag_var, time, lat, lon] --> [time, diag_var, lat, lon]
                            y_diag = y_diag.permute(1, 0, 2, 3)
    
                        # =============================================== #
                        # separate single variable vs. single history_len
                        elif len(y_diag.shape) == 3:
                            if len(self.varname_diagnostic) > 1:
                                # single time, multi-vars
                                y_diag = y_diag.unsqueeze(0)
                            else:
                                # multi-time, single vars
                                y_diag = y_diag.unsqueeze(1)
                        # =============================================== #
    
                        else:
                            # num_var=1, time=1, only has lat, lon
                            y_diag = y_diag.unsqueeze(0).unsqueeze(0)
    
                        return_dict["y_diag"] = y_diag.type(self.output_dtype)
    
                    if self.flag_surface:
                        return_dict["y_surf"] = x_surf.type(self.output_dtype)
    
                    return_dict["y"] = x_upper_air.type(self.output_dtype)

            # ================================================================= #
            # boundary domain tensor conversion
            # ================================================================= #
            elif key == 'boundary_input':

                # upper air boundary inputs
                x_upper_air_outside = np.hstack(
                    [np.expand_dims(var_upper_air_outside, axis=1) for var_upper_air_outside in numpy_vars_upper_air_outside]
                )
                
                x_upper_air_outside = torch.as_tensor(x_upper_air_outside)
                return_dict["x_boundary"] = x_upper_air_outside.type(self.output_dtype)
                
                # surface boundary inputs
                if self.flag_surface_outside:
                    # this line produces [surface_var, time, lat, lon]
                    x_surf_outside = torch.as_tensor(numpy_vars_surface_outside).squeeze()
    
                    if len(x_surf_outside.shape) == 4:
                        # permute: [surface_var, time, lat, lon] --> [time, surface_var, lat, lon]
                        x_surf_outside = x_surf_outside.permute(1, 0, 2, 3)
                        
                    # separate single variable vs. single history_len
                    elif len(x_surf_outside.shape) == 3:
                        if len(self.varname_surface_outside) > 1:
                            # single time, multi-vars
                            x_surf_outside = x_surf_outside.unsqueeze(0)
                        else:
                            # multi-time, single vars
                            x_surf_outside = x_surf_outside.unsqueeze(1)
                    else:
                        # num_var=1, time=1, only has lat, lon
                        x_surf_outside = x_surf_outside.unsqueeze(0).unsqueeze(0)
                        
                    return_dict["x_surf_boundary"] = x_surf_outside.type(self.output_dtype)
                    
            elif key == 'time_encode':
                return_dict["x_time_encode"] = torch.as_tensor(value).type(self.output_dtype)
                
        return return_dict

## boundary and interior data

In [5]:
ds_ERA5 = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/all_in_one/ERA5_GP_1980.zarr')

In [7]:
ds_ERA5

<xarray.Dataset>
Dimensions:    (time: 1464, latitude: 88, longitude: 112, level: 11)
Coordinates:
  * latitude   (latitude) float64 45.0 44.75 44.5 44.25 ... 23.75 23.5 23.25
  * level      (level) float64 50.0 100.0 200.0 300.0 ... 850.0 950.0 1e+03
  * longitude  (longitude) float64 250.2 250.5 250.8 251.0 ... 277.5 277.8 278.0
  * time       (time) datetime64[ns] 1980-01-01 ... 1980-12-31T18:00:00
Data variables:
    MSL        (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    Q          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    SP         (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    T          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    U          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    V          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    VAR_10U    (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    VAR_10V    (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    VAR_2T     (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    Z          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
Attributes:
    CONVERSION_DATE:      Sun May 19 19:53:54 MDT 2019
    CONVERSION_PLATFORM:  Linux r1i0n9 3.12.62-60.64.8-default #1 SMP Tue Oct...
    Conventions:          CF-1.6
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NCO:                  netCDF Operators version 4.7.4 (http://nco.sf.net)
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    history:              Sun May 19 19:54:09 2019: ncks -4 --ppc default=7 e...

In [6]:
ds_C404 = xr.open_zarr('/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/all_in_one/C404_GP_1980.zarr')

In [8]:
ds_C404

<xarray.Dataset>
Dimensions:                    (time: 8784, south_north: 336, west_east: 336,
                                bottom_top: 12)
Coordinates:
  * time                       (time) datetime64[ns] 1980-01-01 ... 1980-12-3...
Dimensions without coordinates: south_north, west_east, bottom_top
Data variables: (12/14)
    WRF_MSLP                   (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_Q                      (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(1, 12, 336, 336), meta=np.ndarray>
    WRF_SP                     (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_T                      (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(1, 12, 336, 336), meta=np.ndarray>
    WRF_T2                     (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_TD2                    (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    ...                         ...
    WRF_V                      (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(1, 12, 336, 336), meta=np.ndarray>
    WRF_V10                    (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_column_total_moisture  (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_evapor                 (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_precip                 (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_radar_composite        (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>

In [7]:
ds_static = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/static/C404_GP_static.zarr')

In [8]:
ds_static

<xarray.Dataset>
Dimensions:   (south_north: 336, west_east: 336)
Dimensions without coordinates: south_north, west_east
Data variables:
    HGT_M     (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    LANDMASK  (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    XLAT      (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    XLONG     (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    z_norm    (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
Attributes: (12/47)
    BOTTOM-TOP_GRID_DIMENSION:       0
    CEN_LAT:                         39.100006103515625
    CEN_LON:                         -97.89999389648438
    DX:                              4000.0
    DY:                              4000.0
    DYN_OPT:                         2
    ...                              ...
    j_parent_end:                    1016
    j_parent_start:                  1
    parent_grid_ratio:               1
    parent_id:                       1
    sr_x:                            1
    sr_y:                            1